In [16]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, types # Export to DataBase

# read XPT file
df_exam_BMI = pd.read_sas("../data/examination_data/BMX_L.xpt", format="xport", encoding="utf-8")

# show top 5 rows 
df_exam_BMI.head()

,SEQN,BMDSTATS,BMXWT,BMIWT,BMXRECUM,BMIRECUM,BMXHEAD,BMIHEAD,BMXHT,BMIHT,...,BMXLEG,BMILEG,BMXARML,BMIARML,BMXARMC,BMIARMC,BMXWAIST,BMIWAIST,BMXHIP,BMIHIP
0,130378.0,1.0,86.9,NaN,NaN,NaN,NaN,NaN,179.5,NaN,...,42.8,NaN,42.0,NaN,35.7,NaN,98.3,NaN,102.9,NaN
1,130379.0,1.0,101.8,NaN,NaN,NaN,NaN,NaN,174.2,NaN,...,38.5,NaN,38.7,NaN,33.7,NaN,114.7,NaN,112.4,NaN
2,130380.0,1.0,69.4,NaN,NaN,NaN,NaN,NaN,152.9,NaN,...,38.5,NaN,35.5,NaN,36.3,NaN,93.5,NaN,98.0,NaN
3,130381.0,1.0,34.3,NaN,NaN,NaN,NaN,NaN,120.1,NaN,...,NaN,NaN,25.4,NaN,23.4,NaN,70.4,NaN,NaN,NaN
4,130382.0,3.0,13.6,NaN,NaN,1.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,1.0,NaN,1.0,NaN,1.0,NaN,NaN


In [17]:
print("BMI Examination Data Info:")
df_exam_BMI.info() 

BMI Examination Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8860 entries, 0 to 8859
Data columns (total 22 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      8860 non-null   float64
 1   BMDSTATS  8860 non-null   float64
 2   BMXWT     8754 non-null   float64
 3   BMIWT     345 non-null    float64
 4   BMXRECUM  454 non-null    float64
 5   BMIRECUM  18 non-null     float64
 6   BMXHEAD   70 non-null     float64
 7   BMIHEAD   0 non-null      float64
 8   BMXHT     8499 non-null   float64
 9   BMIHT     134 non-null    float64
 10  BMXBMI    8471 non-null   float64
 11  BMDBMIC   2492 non-null   float64
 12  BMXLEG    7335 non-null   float64
 13  BMILEG    396 non-null    float64
 14  BMXARML   8568 non-null   float64
 15  BMIARML   200 non-null    float64
 16  BMXARMC   8562 non-null   float64
 17  BMIARMC   205 non-null    float64
 18  BMXWAIST  8190 non-null   float64
 19  BMIWAIST  347 non-null    float64
 20  BMX

In [18]:
df_exam_BMI.columns

Index(['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD',
       'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG',
       'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST',
       'BMXHIP', 'BMIHIP'],
      dtype='object')

In [19]:
# Select and rename essential columns
    # We'll keep SEQN for merging

BMI_cols_to_keep_and_rename = {
    'SEQN': 'Participant_ID',
    'BMXWT': 'Weight_kg',             
    'BMXHT': 'Height_cm',            
    'BMXBMI': 'BMI',                  # Body Mass Index (kg/m^2)
    'BMXWAIST': 'Waist_Circumference_cm', 
    'BMXHIP': 'Hip_Circumference_cm',   
}

df_exam_BMI_selected = df_exam_BMI[list(BMI_cols_to_keep_and_rename.keys())].copy() # list(...) transfers dicts into lists, which then can be worked in dataframe. 
df_exam_BMI_selected.rename(columns=BMI_cols_to_keep_and_rename, inplace=True) 

print("--- Selected and Renamed Blood Pressure Examination Data Info ---")
df_exam_BMI_selected.info()

--- Selected and Renamed Blood Pressure Examination Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8860 entries, 0 to 8859
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Participant_ID          8860 non-null   float64
 1   Weight_kg               8754 non-null   float64
 2   Height_cm               8499 non-null   float64
 3   BMI                     8471 non-null   float64
 4   Waist_Circumference_cm  8190 non-null   float64
 5   Hip_Circumference_cm    6776 non-null   float64
dtypes: float64(6)
memory usage: 415.4 KB


In [20]:
df_exam_BMI_selected

,Participant_ID,Weight_kg,Height_cm,BMI,Waist_Circumference_cm,Hip_Circumference_cm
0,130378.0,86.9,179.5,27.0,98.3,102.9
1,130379.0,101.8,174.2,33.5,114.7,112.4
2,130380.0,69.4,152.9,29.7,93.5,98.0
3,130381.0,34.3,120.1,23.8,70.4,NaN
4,130382.0,13.6,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
8855,142306.0,25.3,128.0,15.4,57.7,NaN
8856,142307.0,NaN,143.8,NaN,NaN,NaN
8857,142308.0,79.3,173.3,26.4,98.4,97.7
8858,142309.0,81.9,179.1,25.5,96.0,103.3


In [ ]:
# Classify BMI Category # https://apps.who.int/nutrition/landscape/help.aspx?menu=0&helpid=420
def classify_bmi(bmi):
    '''this function implements the standard WHO BMI classifications.'''
    if pd.isna(bmi):
        return pd.NA
    elif bmi < 18.5:
        return 'underweight'
    elif bmi < 25:
        return 'normal'
    elif bmi < 30:
        return 'overweight'
    else:
        return 'obese'

df_exam_BMI_selected['BMI_Category'] = df_exam_BMI_selected['BMI'].apply(classify_bmi)
df_exam_BMI_selected

,Participant_ID,Weight_kg,Height_cm,BMI,Waist_Circumference_cm,Hip_Circumference_cm,BMI_Category
0,130378.0,86.9,179.5,27.0,98.3,102.9,overweight
1,130379.0,101.8,174.2,33.5,114.7,112.4,obese
2,130380.0,69.4,152.9,29.7,93.5,98.0,overweight
3,130381.0,34.3,120.1,23.8,70.4,NaN,normal
4,130382.0,13.6,NaN,NaN,NaN,NaN,<NA>
...,...,...,...,...,...,...,...
8855,142306.0,25.3,128.0,15.4,57.7,NaN,underweight
8856,142307.0,NaN,143.8,NaN,NaN,NaN,<NA>
8857,142308.0,79.3,173.3,26.4,98.4,97.7,overweight
8858,142309.0,81.9,179.1,25.5,96.0,103.3,overweight


In [ ]:
# Calculate WHR (Waist-to-Hip Ratio)

df_demo_cleaned = pd.read_csv("../data/demo_data/cleaned_demographics_data.csv")
df_gender = df_demo_cleaned[['Participant_ID', 'Gender']]

df_exam_BMI_selected = df_exam_BMI_selected.merge(df_gender, on='Participant_ID', how='left')

# WHR = Waist Circumference (cm) / Hip Circumference (cm)
df_exam_BMI_selected['WHR'] = (df_exam_BMI_selected['Waist_Circumference_cm'] / df_exam_BMI_selected['Hip_Circumference_cm']).round(1)

# Classify abdominal obesity based on WHR + Gender
    # https://www.cambridge.org/core/journals/public-health-nutrition/article/body-mass-index-waist-circumference-and-waisttohip-ratio-cutoffpoints-for-categorisation-of-obesity-among-omani-arabs/25869FA95B3EFAC462CD94BF386B7885#:~:text=The%20World%20Health%20Organization%20%28WHO%29%20defines%20overweight%20as,%E2%89%A50.90%20in%20men%20and%20%E2%89%A50.85%20in%20women%202.
def classify_abdominal_obesity(row):
    '''This function implements the standard World Health Organization (WHO) cutoffs for abdominal obesity based on WHR and gender'''
    if pd.isna(row['WHR']) or pd.isna(row['Gender']):
        return pd.NA
    if row['Gender'] == 'male' and row['WHR'] >= 0.90:
        return True
    elif row['Gender'] == 'female' and row['WHR'] >= 0.85:
        return True
    else:
        return False

df_exam_BMI_selected['Abdominal_Obesity'] = df_exam_BMI_selected.apply(classify_abdominal_obesity, axis=1)

In [23]:
df_exam_BMI_selected['Participant_ID'] = pd.to_numeric(df_exam_BMI_selected['Participant_ID'], errors='coerce').round().astype('Int64')
df_exam_BMI_selected

,Participant_ID,Weight_kg,Height_cm,BMI,Waist_Circumference_cm,Hip_Circumference_cm,BMI_Category,Gender,WHR,Abdominal_Obesity
0,130378,86.9,179.5,27.0,98.3,102.9,overweight,male,1.0,True
1,130379,101.8,174.2,33.5,114.7,112.4,obese,male,1.0,True
2,130380,69.4,152.9,29.7,93.5,98.0,overweight,female,1.0,True
3,130381,34.3,120.1,23.8,70.4,NaN,normal,female,NaN,<NA>
4,130382,13.6,NaN,NaN,NaN,NaN,<NA>,male,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...
8855,142306,25.3,128.0,15.4,57.7,NaN,underweight,male,NaN,<NA>
8856,142307,NaN,143.8,NaN,NaN,NaN,<NA>,female,NaN,<NA>
8857,142308,79.3,173.3,26.4,98.4,97.7,overweight,male,1.0,True
8858,142309,81.9,179.1,25.5,96.0,103.3,overweight,male,0.9,True


In [24]:
df_exam_BMI_selected.insert(1, 'Gender', df_exam_BMI_selected.pop('Gender'))
df_exam_BMI_selected

,Participant_ID,Gender,Weight_kg,Height_cm,BMI,Waist_Circumference_cm,Hip_Circumference_cm,BMI_Category,WHR,Abdominal_Obesity
0,130378,male,86.9,179.5,27.0,98.3,102.9,overweight,1.0,True
1,130379,male,101.8,174.2,33.5,114.7,112.4,obese,1.0,True
2,130380,female,69.4,152.9,29.7,93.5,98.0,overweight,1.0,True
3,130381,female,34.3,120.1,23.8,70.4,NaN,normal,NaN,<NA>
4,130382,male,13.6,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...
8855,142306,male,25.3,128.0,15.4,57.7,NaN,underweight,NaN,<NA>
8856,142307,female,NaN,143.8,NaN,NaN,NaN,<NA>,NaN,<NA>
8857,142308,male,79.3,173.3,26.4,98.4,97.7,overweight,1.0,True
8858,142309,male,81.9,179.1,25.5,96.0,103.3,overweight,0.9,True


In [25]:
print("BMI examination Data Info - cleaned:")
df_exam_BMI_selected.info() 

BMI examination Data Info - cleaned:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8860 entries, 0 to 8859
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Participant_ID          8860 non-null   Int64  
 1   Gender                  8860 non-null   object 
 2   Weight_kg               8754 non-null   float64
 3   Height_cm               8499 non-null   float64
 4   BMI                     8471 non-null   float64
 5   Waist_Circumference_cm  8190 non-null   float64
 6   Hip_Circumference_cm    6776 non-null   float64
 7   BMI_Category            8471 non-null   object 
 8   WHR                     6764 non-null   float64
 9   Abdominal_Obesity       6764 non-null   object 
dtypes: Int64(1), float64(6), object(3)
memory usage: 701.0+ KB


In [26]:
# export as csv
file_path = "../data/examination_data/cleaned_BMI_examination_data.csv" 

try:
    df_exam_BMI_selected.to_csv(file_path, index=False, encoding='utf-8')
    print(f"DataFrame successfully saved to: {file_path}")
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")

DataFrame successfully saved to: ../data/examination_data/cleaned_BMI_examination_data.csv


In [27]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 

In [28]:
df_exam_BMI_selected.to_sql(name = 'BMI_exmaination_data', con=engine, schema='capstone_group_3',if_exists='replace',index=False)

860